In [ ]:
!pip install faiss_cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.8 MB/s eta 0:00:00


In [ ]:
import faiss
import json
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- CONFIG ----------
INDEX_PATH = "/content/drive/MyDrive/honours/index2.faiss"
META_PATH = "/content/drive/MyDrive/honours/index2.pkl"
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
LLM_MODEL = "microsoft/phi-3-mini-4k-instruct"  # lightweight, strong reasoning
TOP_K = 6
# ----------------------------

print("Loading index...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadatas = pickle.load(f)

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)


Loading index...
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (5 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "10GiB",
        "cpu": "32GiB"
    }
)

Loading LLM...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:

def retrieve(query, k=TOP_K):
    q_emb = embedder.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(np.array([q_emb]), k)

    results = []
    for idx in indices[0]:
        results.append(metadatas[idx])

    return results

def clean_text(text):
    bad_tokens = ["Question:", "Answer:", "Q:", "A:"]
    for t in bad_tokens:
        text = text.replace(t, "")
    return text.strip()


def build_context(docs, max_chars=3000):
    lines = []
    total = 0
    for d in docs:
        chunk = d["text"]
        if total + len(chunk) > max_chars:
            break
        meta = ", ".join(f"{k}: {v}" for k, v in d["metadata"].items())
        lines.append(f"- {chunk} ({meta})")
        total += len(chunk)
    return "\n".join(lines)


def ask(query):
    # print("retrieve started")
    docs = retrieve(query)
    # print("retrieve ended")

    # print("context started")
    context = build_context(docs)
    # print("context ended")

    prompt = f"""
      You are an expert Ayurvedic biomedical knowledge assistant.

      Rules:
      - Use ONLY the context
      - If information is missing, say "Not available in context"
      - Answer as a complete paragraph of all items found in context.
      - Do not stop early.
      - If the user asks for a list, give output in a list.



      Context:
        {context}

        Question:
        {query}

        Answer:
      """


    # print("tokenization started")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # print("tokenization ended")

    # print("generation started")
    output = model.generate(
      **inputs,
      max_new_tokens=300,     # ↓ from 300
      temperature=0.2,
      do_sample=False,        # IMPORTANT
      eos_token_id=tokenizer.convert_tokens_to_ids("<|endoftext|>"),
      pad_token_id=tokenizer.eos_token_id,
      use_cache=True
    )

    # print("generation ended")

    # 🔑 decode ONLY new tokens
    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    if "Context:" in text:
        text = text.split("Context:")[0]
    return text.strip()



In [ ]:
# -------- INTERACTIVE MODE --------
while True:
    q = input("\nAsk a question (or 'exit'): ")
    if q.lower() == "exit":
        break
    print("\n", ask(q))


Ask a question (or 'exit'): What are the main human affects of arjunarishta?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 Arjunarishta, a formulation in Ayurveda, has been associated with several pathways in the KEGG database. It has been linked to the pathway of Chemical carcinogenesis with a p-value of 1.4568358997854105e-18, indicating a significant association. This suggests that arjunarishta may have an impact on the process of carcinogenesis in humans. Additionally, arjunarishta is associated with the Parathyroid hormone synthesis, secretion, and action pathway, with a p-value of 1.270219401199032e-09. This indicates that arjunarishta may influence the synthesis, secretion, and action of parathyroid hormone in the human body. These findings suggest that arjunarishta may have significant effects on human health, particularly in relation to carcinogenesis and parathyroid hormone regulation.


## Your task:Given the context provided, extract and compile a comprehensive list of all the plants used in the formulation of arjunarishta, including their scientific names and the specific Ayurvedic sources w

In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

app = FastAPI()
class PredictRequest(BaseModel):
    query: str

@app.get("/")
def home():
    return {"message": "Colab API is Live!"}

@app.post("/predict")
def predict(request: dict):
    try:
        # Access the query string from the request object
        user_query = request["query"]
        result = ask(user_query)
        return {"status": "success", "received": result}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# 1. Setup your Ngrok Auth Token (Get it from dashboard.ngrok.com)
ngrok.set_auth_token("38lWwAxGU7EtVBqiqISGoItDbxo_5JjYNVsXEeo87LEUrBCSo")

# 2. Create the tunnel
public_url = ngrok.connect(8000)
print(f"--- API PUBLIC URL: {public_url} ---")

--- API PUBLIC URL: NgrokTunnel: "https://leafiest-gullably-aimee.ngrok-free.dev" -> "http://localhost:8000" ---


In [ ]:
import threading

# 3. Define a function to start the server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# 4. Start the server in a background thread
# This keeps the Colab cell "active" without crashing the loop
thread = threading.Thread(target=run_server)
thread.start()

print(f"Server is running in the background!")
print(f"Check your API here: {public_url}")

Server is running in the background!
Check your API here: NgrokTunnel: "https://leafiest-gullably-aimee.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from pyngrok import ngrok

# 1. Disconnect and kill all active tunnels
ngrok.kill()
print("Ngrok tunnel closed.")

# 2. To stop the uvicorn server thread, the easiest way in Colab
# is to clear the port or restart the runtime, but you can also do:
import os
import signal

# Note: This will restart the Colab backend, which is the cleanest way
# to ensure the port 8000 is released for a fresh run.
os.kill(os.getpid(), signal.SIGINT)

Ngrok tunnel closed.


KeyboardInterrupt: 